# DistilBERT router training on a virtual GPU

This notebook fine-tunes the repository trainer on **private, already-prepared development labels**. It does not provision a GPU, call model providers, create labels, or inspect the held-out test split. In Colab, choose **Runtime → Change runtime type → GPU** before running the cells.

## Readiness gate

The current project audit produced **0/1,500 train labels and 0/500 validation labels** for the selected three-tier recorded-cost target: frontier observations are missing, and the local observations have unknown charges. Training cannot start until you have genuine small, medium, and frontier results with known comparable costs for the development splits and have prepared nonempty train and validation label JSONL files with the repository's `prepare` command. Do not replace unknown local costs with zero or change the target to two tiers.

The notebook deliberately stops if those files are absent, incomplete, overlapping, or do not contain all three classes. Keep raw benchmark runs, prompt text, and label files private; upload them to this temporary runtime only. **Do not commit them or add them to the source archive.**

Training defaults are seed 17, 3 epochs, max length 256, learning rate 2e-5, and batch size 16. The notebook records the actual runtime and parameters in the exported artifact. GPU allocation, runtime duration, and final quality are only known after you run it.


## 1. Pin the source you are training

From your project checkout, create a zip of a committed revision and record the two values below. The source archive must include the trainer change that identifies the base model as `distilbert/distilbert-base-uncased`.

```bash
git rev-parse HEAD
git archive --format=zip -o laya-gateway-source.zip HEAD
shasum -a 256 laya-gateway-source.zip
```

Paste the 40-character Git revision and 64-character SHA-256 below. Upload the zip when prompted. The archive is extracted only after its digest and safe paths are checked. The model revision is pinned to Hugging Face commit `12040accade4e8a0f71eabdb258fecc2e7e948be` (the verified May 2024 model repository commit).

In [ ]:
from pathlib import Path
import hashlib
import re
import zipfile

SOURCE_REVISION = ""
SOURCE_ARCHIVE_SHA256 = ""
MODEL_ID = "distilbert/distilbert-base-uncased"
MODEL_REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"

assert re.fullmatch(r"[0-9a-f]{40}", SOURCE_REVISION), "Paste the full `git rev-parse HEAD` value."
assert re.fullmatch(r"[0-9a-f]{64}", SOURCE_ARCHIVE_SHA256), "Paste the SHA-256 of the uploaded source zip."
assert re.fullmatch(r"[0-9a-f]{40}", MODEL_REVISION), "Model revision must be a full immutable commit hash."
assert MODEL_ID == "distilbert/distilbert-base-uncased"

from google.colab import files
uploaded = files.upload()
archive_name = "laya-gateway-source.zip"
assert archive_name in uploaded, f"Upload a file named {archive_name}."
archive_path = Path("/content") / archive_name
actual_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
assert actual_sha256 == SOURCE_ARCHIVE_SHA256, "Source archive digest does not match the value you pasted."

source_dir = Path("/content/laya-gateway-source")
source_dir.mkdir(exist_ok=False)
with zipfile.ZipFile(archive_path) as archive:
    for item in archive.infolist():
        target = (source_dir / item.filename).resolve()
        assert target == source_dir.resolve() or source_dir.resolve() in target.parents, f"Unsafe archive path: {item.filename}"
    archive.extractall(source_dir)
assert (source_dir / "pyproject.toml").is_file()
assert (source_dir / "src/laya_gateway/routing/train.py").is_file()
print({"source_revision": SOURCE_REVISION, "source_sha256": actual_sha256, "source_dir": str(source_dir)})

## 2. Check the allocated GPU and install training packages

This checks CUDA **before** installation and confirms the existing Torch build is unchanged afterward. It refuses to continue on a CPU-only runtime. The installation does not request a Torch package, so Colab's CUDA-enabled Torch is preserved.

In [ ]:
import json
import subprocess
import sys

preflight = subprocess.run([sys.executable, str(source_dir / "notebooks/colab_preflight.py")],
                           check=True, text=True, capture_output=True)
print(preflight.stdout)

import torch
assert torch.cuda.is_available(), "No CUDA GPU is attached. In Colab choose Runtime → Change runtime type → GPU, reconnect, then rerun."
torch_before = torch.__version__
gpu_name = torch.cuda.get_device_name(0)
print({"torch": torch_before, "cuda": torch.version.cuda, "gpu": gpu_name,
       "gpu_memory_bytes": torch.cuda.get_device_properties(0).total_memory})

subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(source_dir)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install",
                "httpx>=0.27,<1", "pydantic>=2.10,<3", "transformers>=4.50,<5",
                "datasets>=3,<5", "accelerate>=1,<2"], check=True)
import torch as torch_after_install
assert torch_after_install.__version__ == torch_before, "Torch changed during installation; stop and inspect the runtime."
assert torch_after_install.cuda.is_available(), "CUDA became unavailable after installation."
print({"training_dependencies": "installed", "torch_unchanged": True,
       "transformers": __import__("transformers").__version__,
       "datasets": __import__("datasets").__version__,
       "accelerate": __import__("accelerate").__version__})

## 3. Upload and validate private train/validation labels

Prepare the files locally only after all three tiers have real results and comparable recorded charges. Use the repository's private manifest and run files; pass the three run files for each split, including the saved medium validation rescore when applicable. The command prints aggregate label counts and writes only under ignored `runs/`:

```bash
.venv-distilbert/bin/python -m laya_gateway.routing.prepare \
  --manifest runs/day2-manifest.jsonl \
  --run runs/SMALL_TRAIN.jsonl --run runs/MEDIUM_TRAIN.jsonl --run runs/FRONTIER_TRAIN.jsonl \
  --split train --output runs/distilbert-train-labels.jsonl

.venv-distilbert/bin/python -m laya_gateway.routing.prepare \
  --manifest runs/day2-manifest.jsonl \
  --run runs/SMALL_VALIDATION.jsonl --run runs/MEDIUM_VALIDATION.jsonl --run runs/FRONTIER_VALIDATION.jsonl \
  --split validation --output runs/distilbert-validation-labels.jsonl
```

Replace the example paths with the actual immutable run files. Do not combine duplicate versions of one tier; use the accepted rescore in place of its failed original result. Both files must be nonempty and contain all three labels. The upload check prints counts and fingerprints only, never prompt text.

In [ ]:
from collections import Counter
import json
from laya_gateway.routing.contracts import TIERS
from laya_gateway.routing.training_data import TrainingRow

uploaded = files.upload()
required_files = {"distilbert-train-labels.jsonl", "distilbert-validation-labels.jsonl"}
assert required_files.issubset(uploaded), f"Upload both files: {sorted(required_files)}"
train_path = Path("/content/distilbert-train-labels.jsonl")
validation_path = Path("/content/distilbert-validation-labels.jsonl")


def read_private_rows(path, expected_split):
    lines = [line for line in path.read_text().splitlines() if line.strip()]
    assert lines, f"{path.name} is empty. Current project labels are not ready for training."
    rows = [TrainingRow(**json.loads(line)) for line in lines]
    assert all(row.split == expected_split for row in rows), f"{path.name} includes the wrong split."
    assert all(row.label in TIERS for row in rows), "Unexpected label outside the three router tiers."
    assert all(row.cost_basis == "recorded_charge_usd" for row in rows), "Labels do not use recorded comparable charges."
    assert all(set(row.backend_models) == set(TIERS) for row in rows), "Each label must include all three tier model identities."
    assert all(set(row.result_run_ids) == set(TIERS) for row in rows), "Each label must include all three source run IDs."
    assert len({row.prompt_id for row in rows}) == len(rows), f"Duplicate prompt IDs in {path.name}."
    assert set(Counter(row.label for row in rows)) == set(TIERS), f"{path.name} must contain all three label classes."
    return rows

train_rows = read_private_rows(train_path, "train")
validation_rows = read_private_rows(validation_path, "validation")
train_ids = {row.prompt_id for row in train_rows}
validation_ids = {row.prompt_id for row in validation_rows}
assert train_ids.isdisjoint(validation_ids), "Train and validation prompt IDs overlap."
assert len({row.settings_fingerprint for row in train_rows + validation_rows}) == 1, "Settings fingerprints differ."
assert len({tuple(sorted(row.backend_models.items())) for row in train_rows + validation_rows}) == 1, "Backend model identities differ."
print(json.dumps({"train_rows": len(train_rows), "train_label_counts": dict(Counter(r.label for r in train_rows)),
                  "validation_rows": len(validation_rows), "validation_label_counts": dict(Counter(r.label for r in validation_rows)),
                  "settings_fingerprint": train_rows[0].settings_fingerprint,
                  "backend_models": train_rows[0].backend_models}, indent=2, sort_keys=True))

## 4. Train, verify the saved artifact, and download it

This cell uses the existing project trainer and creates a new run directory. With roughly 1,500 train and 500 validation prompts, total time depends on the virtual GPU, tokenizer/model download, and Colab load. Keep this tab open until the archive downloads. If the GPU runs out of memory, stop and record that failure before making a separate run with a smaller batch size; do not silently alter this run.

In [ ]:
from datetime import datetime, timezone
import shutil
import uuid

SEED = 17
MAX_LENGTH = 256
EPOCHS = 3
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
output_dir = Path("/content/distilbert-runs") / f"distilbert-{run_id}"
output_dir.parent.mkdir(parents=True, exist_ok=True)
assert not output_dir.exists()

subprocess.run([sys.executable, "-m", "laya_gateway.routing.train",
                "--train", str(train_path), "--validation", str(validation_path),
                "--output", str(output_dir), "--revision", MODEL_REVISION,
                "--seed", str(SEED), "--max-length", str(MAX_LENGTH),
                "--epochs", str(EPOCHS), "--learning-rate", str(LEARNING_RATE),
                "--batch-size", str(BATCH_SIZE)], check=True)

from laya_gateway.routing.artifact import inspect
metadata = inspect(output_dir)
assert metadata["base_model"] == MODEL_ID and metadata["base_revision"] == MODEL_REVISION
summary = {"run_id": run_id, "source_revision": SOURCE_REVISION,
           "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
           "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
           "gpu": gpu_name, "torch": torch.__version__, "cuda": torch.version.cuda,
           "seed": SEED, "max_length": MAX_LENGTH, "epochs": EPOCHS,
           "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE,
           "train_counts": metadata["train_counts"],
           "validation_counts": metadata["validation_counts"],
           "evaluation": metadata["evaluation"],
           "artifact_reload_check": json.loads((output_dir / "reload_check.json").read_text())}
(output_dir / "virtual_gpu_run.json").write_text(json.dumps(summary, indent=2, sort_keys=True))

artifact_zip = Path("/content") / f"{output_dir.name}.zip"
shutil.make_archive(str(artifact_zip.with_suffix("")), "zip", root_dir=output_dir.parent,
                    base_dir=output_dir.name)
print(json.dumps(summary, indent=2, sort_keys=True))
print({"download_file": str(artifact_zip), "artifact_bytes": artifact_zip.stat().st_size})
files.download(str(artifact_zip))

## After training

Keep the downloaded artifact and the printed `run_id` with your private experiment records. The artifact contains model weights, tokenizer, trainer metadata, a CPU reload check, and sanitized GPU/runtime summary. The validation score is for model selection; it is not the held-out test result. Do not publish the weights or benchmark data without separately reviewing the data and model licensing and your intended destination.